In [25]:
#1. Реализация n-граммного теггера
import nltk
from nltk.corpus import PlaintextCorpusReader, treebank
from nltk.tag import UnigramTagger, BigramTagger, TrigramTagger

with open('motor.txt' , 'r') as f:
  my_file = f.read()

# Загружаем корпус
nltk.download('treebank')
corpus = treebank.tagged_sents()

# Функция для создания n-граммного теггера
def train_tagger(corpus):
    unigram_tagger = UnigramTagger(corpus)
    bigram_tagger = BigramTagger(corpus, backoff=unigram_tagger)
    trigram_tagger = TrigramTagger(corpus, backoff=bigram_tagger)
    return trigram_tagger

tagger = train_tagger(corpus)

# Тестирование
tokens = nltk.word_tokenize(my_file)
print(tagger.tag(tokens))

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package treebank is already up-to-date!


[('McLaren', None), ('insists', 'VBZ'), ('Norris', None), ('title', None), ('was', 'VBD'), ('never', 'RB'), ('main', 'JJ'), ('goal', 'NN'), (',', ','), ('after', 'IN'), ('Brazil', 'NNP'), ('setback', 'NN'), ('While', 'IN'), ('Lando', None), ('NorrisвЂ™s', None), ('hopes', 'VBZ'), ('of', 'IN'), ('winning', 'VBG'), ('the', 'DT'), ('driversвЂ™', None), ('crown', None), ('are', 'VBP'), ('all', 'DT'), ('but', 'CC'), ('over', 'IN'), (',', ','), ('McLaren', None), ('says', 'VBZ'), ('its', 'PRP$'), ('main', 'JJ'), ('ambition', None), ('remains', 'VBZ'), ('intact', None), ('McLaren', None), ('says', 'VBZ'), ('that', 'IN'), ('guiding', None), ('Lando', None), ('Norris', None), ('to', 'TO'), ('the', 'DT'), ('driversвЂ™', None), ('championship', None), ('was', 'VBD'), ('never', 'RB'), ('ultimately', 'RB'), ('its', 'PRP$'), ('main', 'JJ'), ('target', 'NN'), ('вЂ', None), ('“', None), ('as', 'IN'), ('it', 'PRP'), ('has', 'VBZ'), ('always', 'RB'), ('been', 'VBN'), ('more', 'JJR'), ('focused', 'VBN'),

In [ ]:
#2. Сравнение эффективности двух методик снятия лексической и морфологической неоднозначности
import nltk
from nltk.tag import UnigramTagger
import pymorphy3

# Установка ресурсов NLTK (при необходимости)
nltk.download('punkt')

# Инициализация морфологического анализатора pymorphy3
morph = pymorphy3.MorphAnalyzer()

# 2. Функция для морфологической разметки pymorphy3
def pymorphy3_tag(text):
    tokens = nltk.word_tokenize(text)
    tagged = []
    for word in tokens:
        parsed = morph.parse(word)[0]  # Берем первый разбор как наиболее вероятный
        tagged.append((word, parsed.tag.POS))  # Извлекаем только POS-тег
    return tagged

# 3. Функция для подготовки NLTK-теггера
def train_nltk_tagger(corpus):
    unigram_tagger = UnigramTagger(corpus)
    return unigram_tagger

# Пример корпуса с размеченными предложениями (в формате [(слово, тег), ...])
# Замените на ваш корпус, если он доступен
sample_corpus = [[("Мама", "NOUN"), ("мыла", "VERB"), ("раму", "NOUN")]] 
nltk_tagger = train_nltk_tagger(sample_corpus)

# 4. Сравнение результатов pymorphy3 и NLTK-теггера
def evaluate_taggers(text, reference_tags):
    # Разметка pymorphy3
    tagged_pymorphy3 = pymorphy3_tag(text)

    # Разметка NLTK
    tokens = nltk.word_tokenize(text)
    tagged_nltk = nltk_tagger.tag(tokens)

    # Сравнение с эталонной разметкой
    correct_pymorphy3 = sum(1 for ref, test in zip(reference_tags, tagged_pymorphy3) if ref[1] == test[1])
    correct_nltk = sum(1 for ref, test in zip(reference_tags, tagged_nltk) if ref[1] == test[1])
    
    # Вывод результатов
    print(f"Pymorphy3 Accuracy: {correct_pymorphy3 / len(reference_tags):.2%}")
    print(f"NLTK Tagger Accuracy: {correct_nltk / len(reference_tags):.2%}")

# Пример текста и эталонной разметки
text = "Мама мыла раму"
reference_tags = [("Мама", "NOUN"), ("мыла", "VERB"), ("раму", "NOUN")]

# Запуск сравнения
evaluate_taggers(text, reference_tags)


Pymorphy3 Accuracy: 66.67%
NLTK Tagger Accuracy: 100.00%


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# 3. Программа для оценки качества морфологической разметки
def evaluate_morphology(gold_corpus, test_corpus):
    matches, total = 0, 0
    discrepancies = []
    for gold, test in zip(gold_corpus, test_corpus):
        for g, t in zip(gold, test):
            total += 1
            if g == t:
                matches += 1
            else:
                discrepancies.append((g, t))
    # Рассчитываем точность и выводим результаты
    accuracy = matches / total if total > 0 else 0
    print(f"Accuracy: {accuracy:.2%}")
    print("Discrepancies:", discrepancies)

# Пример эталонного и тестового корпусов
gold_corpus = [[("Мама", "NOUN"), ("мыла", "VERB"), ("раму", "NOUN")]]
test_corpus = [[("Мама", "NOUN"), ("мыла", "VERB"), ("раму", "ADJ")]]

# Вызов функции
evaluate_morphology(gold_corpus, test_corpus)



Accuracy: 66.67%
Discrepancies: [(('раму', 'NOUN'), ('раму', 'ADJ'))]


In [ ]:
# 4. Конвертер форматов разметки
import pymorphy3
import nltk

# Инициализация морфологического анализатора pymorphy3
morph = pymorphy3.MorphAnalyzer()

# Функция для морфологического тегирования с pymorphy3
def pymorphy3_tag(text):
    tokens = nltk.word_tokenize(text)
    tagged = []
    for word in tokens:
        parsed = morph.parse(word)[0]  # Используем первый разбор как наиболее вероятный
        tagged.append((word, parsed.tag.POS))  # Извлекаем только POS-тег
    return tagged

# Пример текста и вызов функции
text = "Это пример текста для тестирования теггера."
tagged_text = pymorphy3_tag(text)
print("Tagged text:", tagged_text)




Tagged text: [('Это', 'PRCL'), ('пример', 'NOUN'), ('текста', 'NOUN'), ('для', 'PREP'), ('тестирования', 'NOUN'), ('теггера', 'NOUN'), ('.', None)]


In [ ]:
# 5. Алгоритм для сборки морфологических парадигм по гиперосновам

def generate_paradigm(hyper_stem, paradigm_rules):
    word_forms = []
    for rule in paradigm_rules:
        word_form = apply_rule(hyper_stem, rule)
        word_forms.append(word_form)
    return word_forms

# Пример использования
hyper_stem = "дела"
paradigm_rules = ["ю", "ешь", "ет", "ем", "ете", "ют"]
paradigm = generate_paradigm(hyper_stem, paradigm_rules)
print("Paradigm:", paradigm)


Paradigm: ['делаю', 'делаешь', 'делает', 'делаем', 'делаете', 'делают']
